# Fitting the national models

Refits the six production models from scratch and checks them against the pickles the published
estimates are served from. The fitting logic lives in `src/modelling.py`; this notebook chooses the
inputs, runs it and reads the result.

**This does not change any published number.** `13_coherent_ggi` reads `dgg_pipeline`'s golden
series, not these pickles (D16), so a refit here is a check, never a release. Promoting a refit is
a deliberate act — see `doc/modelling.md` §6.

In [1]:
import _bootstrap  # noqa: F401  — puts src/ on sys.path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import params
import plotting as plot
import modelling

CFG = params.FINAL_MODEL
print('production variant :', CFG['model_type'])
print('spec               :', ' + '.join(CFG['spec']))
print('dataset            :', CFG['dataset'])
print('held-out unit      :', CFG['leave_column'])

production variant : combined_with_CIS
spec               : fb_18_999_men + fb_18_999_wom + fb_18_999_r + hdi + gdi + gdp_pcap + year
dataset            : combined_multiple_years_no_missing_keep_countries_fb_aligned_itu_deleted.csv
held-out unit      : iso3


## 1. The fitting panel

One row per country-survey, so a country with three surveys contributes three. `keep_itu` is the
whole of the sample difference between model variants — despite the file's `_itu_deleted` name it
carries ITU rows (D37).

In [2]:
for indicator in CFG['indicators']:
    data = modelling.load_training_data(indicator)
    types = data[f'{indicator}_survey_type'].value_counts().to_dict()
    print(f'{indicator:9s} {len(data):4d} rows   {types}')

internet   108 rows   {'itu': 37, 'mics6': 30, 'dhs7': 28, 'dhs8': 10, 'continuous dhs8': 2, 'continuous dhs7': 1}
mobile      99 rows   {'dhs7': 31, 'mics6': 31, 'itu': 24, 'dhs8': 10, 'continuous dhs8': 2, 'continuous dhs7': 1}


## 2. Fit, validate, and compare against the shipped models

For each of the six targets: the OLS fit, the leave-one-country-out predictions, and the
non-negative least-squares fit of |LOCO error| that gives each prediction its uncertainty band.

In [3]:
summary, betas = [], []

for indicator in CFG['indicators']:
    data = modelling.load_training_data(indicator)
    for outcome_var in CFG['outcome_vars']:
        model = modelling.fit_full_model(data, indicator, outcome_var)
        loco = modelling.fit_loco(data, indicator, outcome_var)
        coefficients, stats = modelling.fit_error_betas(loco)

        shipped = utils_shipped = modelling.utils.load_model(
            modelling.shipped_model_path(indicator, outcome_var))
        summary.append({
            'model': f'{indicator}_{outcome_var}', 'n': int(model.nobs),
            'r2': model.rsquared, 'adj_r2': model.rsquared_adj,
            'loco_r2': stats['loco_r2'], 'mean_abs_diff': stats['mean_abs_diff'],
            'max_abs_coef_diff': (float(np.abs(model.params[shipped.params.index]
                                               - shipped.params).max())
                                  if shipped is not None else np.nan),
        })
        betas.append({'model': f'{indicator}_{outcome_var}', 'r2': stats['r2'],
                      **dict(zip(CFG['spec'], coefficients))})

summary = pd.DataFrame(summary)
print(f"refit vs shipped — max |coefficient difference|: "
      f"{summary['max_abs_coef_diff'].max():.2e}")
summary.round(4)

refit vs shipped — max |coefficient difference|: 3.55e-15


,model,n,r2,adj_r2,loco_r2,mean_abs_diff,max_abs_coef_diff
0,internet_ggi,108,0.8281,0.8161,0.7910,0.0841,0.0
1,internet_wom,108,0.9045,0.8979,0.8813,0.0851,0.0
2,internet_men,108,0.8732,0.8643,0.8425,0.0824,0.0
3,mobile_ggi,99,0.7393,0.7192,0.6827,0.0652,0.0
4,mobile_wom,99,0.7959,0.7802,0.7489,0.0829,0.0
5,mobile_men,99,0.6343,0.6062,0.5626,0.0763,0.0


### The error-estimation betas

Non-negative least squares of |LOCO error| on the spec. Worth reading rather than skipping: in the
current fit only `gdi` takes a non-zero coefficient and the fit's own R² is near zero, so the
published uncertainty band is close to a constant times `gdi` (D37).

In [4]:
pd.DataFrame(betas).round(4)

,model,r2,fb_18_999_men,fb_18_999_wom,fb_18_999_r,hdi,gdi,gdp_pcap,year
0,internet_ggi,-0.0234,0.0,0.0,0.0,0.0,0.0883,0.0,0.0
1,internet_wom,-0.0027,0.0,0.0,0.0,0.0,0.0901,0.0,0.0
2,internet_men,0.0005,0.0,0.0,0.0,0.0,0.0875,0.0,0.0
3,mobile_ggi,-0.0335,0.0,0.0,0.0,0.0,0.0689,0.0,0.0
4,mobile_wom,-0.0307,0.0,0.0,0.0,0.0,0.0879,0.0,0.0
5,mobile_men,-0.0117,0.0,0.0,0.0,0.0,0.0814,0.0,0.0


## Notes

- `python src/modelling.py --verify` does the same comparison from the command line and writes
  nothing; without `--verify` it writes a **date-stamped refit directory**, never over the shipped
  models (D14).
- Performance across all ten model variants, and the reproduction of the published performance
  figure, is `03_model_performance/03_01_model_performance.ipynb`.